# Gráficos comparativos SAEB 2021 × 2023 — Dashboard Plotly
# Ensino Básico: 5º ano (série 12) e 9º ano (série 13)

In [2]:
!pip install plotly kaleido -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 2.7 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

In [6]:
PATH_2021 = "SAEB_Tratado_2021.csv"
PATH_2023 = "SAEB_Tratado_2023.csv"

df21 = pd.read_csv(PATH_2021)
df21["ANO"] = 2021


df23 = pd.read_csv(PATH_2023)
df23["ANO"] = 2023

df = pd.concat([df21, df23], ignore_index=True)

In [22]:
REGIOES = {1: "Norte", 2: "Nordeste", 3: "Sudeste", 4: "Sul", 5: "Centro-Oeste"}
ESCOLA  = {1: "Pública", 0: "Privada"}
SERIES  = {12: "5º Ano", 13: "9º Ano"}

df["REGIAO_NOME"]  = df["ID_REGIAO"].map(REGIOES)
df["ESCOLA_TIPO"]  = df["IN_PUBLICA"].map(ESCOLA)
df["SERIE_NOME"]   = df["ID_SERIE"].map(SERIES).fillna(df["ID_SERIE"].astype(str))




###GRÁFICO 1 — Scatter: INSE × Acertos Totais (2021 e 2023)
#####Amostra de 30k pontos por ano para não travar

In [23]:
#pras cores do dashboard
DB_BG       = "#0F1E2E"
DB_CARD     = "#162535"
DB_GRID     = "#1E3448"
DB_TEXT     = "#E2E8F0"
DB_SUBTEXT  = "#94A3B8"
DB_ORANGE   = "#F97316"
DB_FONT     = dict(family="Inter, Arial, sans-serif", size=13, color=DB_TEXT)

#Cores de acento por ano
COR_2021 = "#818CF8"
COR_2023 = "#EAB308"

SAMPLE = 15_000

def scatter_ano(df_ano, ano, cor_pontos, bins=60):
    """Scatter INSE × Acertos"""
    dados = (
        df_ano.dropna(subset=["INSE_ALUNO", "ACERTOS_TOTAIS"])
              .sample(min(SAMPLE, len(df_ano)), random_state=42)
              .copy()
    )

    np.random.seed(42)
    y_jitter = dados["ACERTOS_TOTAIS"] + np.random.uniform(-0.45, 0.45, len(dados))

    #calculo media por faixa inse
    dados["INSE_BIN"] = pd.cut(dados["INSE_ALUNO"], bins=bins)
    media_bin = (
        dados.groupby("INSE_BIN", observed=True)
             .agg(INSE_MED=("INSE_ALUNO", "mean"),
                  ACERTOS_MED=("ACERTOS_TOTAIS", "mean"),
                  N=("ACERTOS_TOTAIS", "count"))
             .dropna()
             .reset_index()
    )

    #regressao linear
    coef = np.polyfit(dados["INSE_ALUNO"], dados["ACERTOS_TOTAIS"], 1)
    x_line = np.linspace(dados["INSE_ALUNO"].min(), dados["INSE_ALUNO"].max(), 100)

    fig = go.Figure()

    #camada 1 nuvem pontos jitter
    fig.add_trace(go.Scatter(
        x=dados["INSE_ALUNO"],
        y=y_jitter,
        mode="markers",
        name="Alunos (amostra)",
        marker=dict(
            color=cor_pontos,
            size=3,
            opacity=0.20,
        ),
        hovertemplate=(
            "<b>Aluno</b><br>"
            "INSE: %{x:.2f}<br>"
            "Acertos: %{customdata}<extra></extra>"
        ),
        customdata=dados["ACERTOS_TOTAIS"].values,
    ))

    #camada 2 curva de media por faixa
    fig.add_trace(go.Scatter(
        x=media_bin["INSE_MED"],
        y=media_bin["ACERTOS_MED"],
        mode="lines",
        name="Média por faixa de INSE",
        line=dict(color=DB_ORANGE, width=3),
        hovertemplate=(
            "Faixa INSE ≈ %{x:.2f}<br>"
            "Média acertos: %{y:.1f}<br>"
            "N: %{customdata}<extra></extra>"
        ),
        customdata=media_bin["N"].values,
    ))

    #camada 3 regressao linear
    fig.add_trace(go.Scatter(
        x=x_line,
        y=np.polyval(coef, x_line),
        mode="lines",
        name=f"Regressão linear",
        line=dict(color="#FFFFFF", width=1.5, dash="dot"),
    ))

    fig.update_layout(

        paper_bgcolor=DB_BG,
        plot_bgcolor=DB_CARD,

        title=dict(
            text=(
                f"<b style='font-size:18px'>Nível Socioeconômico × Desempenho — {ano}</b><br>"
                f"<span style='color:{DB_SUBTEXT};font-size:13px'>"
                f"INSE_ALUNO vs. Acertos Totais (LP + MT) · amostra de {SAMPLE:,} alunos"
                f"</span>"
            ),
            x=0.03, y=0.97,
            font=dict(color=DB_TEXT, family="Inter, Arial, sans-serif"),
        ),

        xaxis=dict(
            title=dict(text="Índice Socioeconômico do Aluno (INSE)", font=dict(color=DB_SUBTEXT, size=12)),
            tickfont=dict(color=DB_SUBTEXT, size=11),
            gridcolor=DB_GRID, gridwidth=1,
            zeroline=False,
            range=[2, 7.8],
            linecolor=DB_GRID,
        ),
        yaxis=dict(
            title=dict(text="Total de Acertos (LP + MT)", font=dict(color=DB_SUBTEXT, size=12)),
            tickfont=dict(color=DB_SUBTEXT, size=11),
            gridcolor=DB_GRID, gridwidth=1,
            zeroline=False,
            range=[-1, 53],
            linecolor=DB_GRID,
        ),

        legend=dict(
            orientation="h",
            y=-0.14,
            x=0,
            font=dict(color=DB_TEXT, size=12),
            bgcolor="rgba(0,0,0,0)",
        ),

        annotations=[dict(
            x=0.98, y=0.06,
            xref="paper", yref="paper",
            text=f"<b>β = {coef[0]:.2f}</b>  acertos / ponto INSE",
            showarrow=False,
            font=dict(color=DB_ORANGE, size=12, family="Inter, Arial, sans-serif"),
            bgcolor=DB_BG,
            bordercolor=DB_ORANGE,
            borderwidth=1,
            borderpad=6,
        )],

        font=DB_FONT,
        height=540,
        margin=dict(l=60, r=40, t=80, b=80),
    )
    return fig

#graficos
fig1a = scatter_ano(df[df["ANO"] == 2021], ano=2021, cor_pontos=COR_2021)
fig1a.show()

fig1b = scatter_ano(df[df["ANO"] == 2023], ano=2023, cor_pontos=COR_2023)
fig1b.show()

##GRÁFICO 2 — Histograma: distribuição de acertos por região

In [19]:
REGIOES_ORDEM = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]
BINS = list(range(0, 53, 1))

def hist_ano(ano, cor_ano):
    fig = make_subplots(
        rows=5, cols=1,
        subplot_titles=[f"<b>{r}</b>" for r in REGIOES_ORDEM],
        shared_xaxes=True,
        vertical_spacing=0.06,
    )

    for row_idx, regiao in enumerate(REGIOES_ORDEM, start=1):
        dados = (
            df[(df["REGIAO_NOME"] == regiao) & (df["ANO"] == ano)]
            ["ACERTOS_TOTAIS"].dropna()
        )
        if dados.empty:
            continue

        counts, edges = np.histogram(dados, bins=BINS)
        pct = counts / counts.sum() * 100
        x_centers = (edges[:-1] + edges[1:]) / 2

        fig.add_trace(go.Bar(
            x=x_centers,
            y=pct,
            name=regiao,
            marker_color=cor_ano,
            opacity=0.70,
            width=0.8,
            showlegend=False,
            hovertemplate=(
                f"<b>{regiao} · {ano}</b><br>"
                "Acertos: %{x}<br>"
                "% alunos: %{y:.2f}%<extra></extra>"
            ),
        ), row=row_idx, col=1)

        media = dados.mean()
        fig.add_vline(
            x=media,
            line_dash="dot",
            line_color=DB_ORANGE,
            line_width=1.5,
            row=row_idx, col=1,
        )

        # Badge com a média no canto de cada painel
        fig.add_annotation(
            x=0.98, y=0.85,
            xref=f"x{row_idx} domain" if row_idx > 1 else "x domain",
            yref=f"y{row_idx} domain" if row_idx > 1 else "y domain",
            text=f"<b>μ = {media:.1f}</b>",
            showarrow=False,
            font=dict(color=DB_ORANGE, size=11, family="Inter, Arial, sans-serif"),
            bgcolor=DB_BG,
            bordercolor=DB_ORANGE,
            borderwidth=1,
            borderpad=4,
        )

    for i in range(1, 6):
        fig.update_yaxes(
            title_text="% alunos" if i == 3 else "",
            gridcolor=DB_GRID,
            ticksuffix="%",
            tickfont=dict(color=DB_SUBTEXT, size=11),
            title_font=dict(color=DB_SUBTEXT, size=12),
            linecolor=DB_GRID,
            row=i, col=1,
        )
    fig.update_xaxes(
        title_text="Total de Acertos (LP + MT)",
        gridcolor=DB_GRID,
        tickfont=dict(color=DB_SUBTEXT, size=11),
        title_font=dict(color=DB_SUBTEXT, size=12),
        linecolor=DB_GRID,
        range=[-0.5, 51.5],
        row=5, col=1,
    )

    for annotation in fig.layout.annotations:
        annotation.font = dict(color=DB_TEXT, size=13, family="Inter, Arial, sans-serif")

    fig.update_layout(
        paper_bgcolor=DB_BG,
        plot_bgcolor=DB_CARD,
        title=dict(
            text=(
                f"<b style='font-size:18px'>Distribuição de Acertos por Região — {ano}</b><br>"
                f"<span style='color:{DB_SUBTEXT};font-size:13px'>"
                "Cada painel = uma região · linha laranja = média</span>"
            ),
            x=0.03, y=0.97,
            font=dict(color=DB_TEXT, family="Inter, Arial, sans-serif"),
        ),
        barmode="overlay",
        font=DB_FONT,
        height=1100,
        margin=dict(l=60, r=40, t=120, b=60),
    )
    return fig

fig2a = hist_ano(2021, COR_2021)
fig2a.show()

fig2b = hist_ano(2023, COR_2023)
fig2b.show()

##GRÁFICO 3 - Barras: INSE médio por região (IN_PUBLICA como filtro)

In [20]:
inse_reg = (
    df.groupby(["ANO", "REGIAO_NOME", "ESCOLA_TIPO"])["INSE_ALUNO"]
      .mean()
      .reset_index()
      .rename(columns={"INSE_ALUNO": "INSE_MEDIO"})
)

CORES_TIPO_DARK = {"Pública": COR_2021, "Privada": COR_2023}

def barras_inse(ano):
    sub = inse_reg[inse_reg["ANO"] == ano].copy()
    sub = sub.sort_values("REGIAO_NOME")

    fig = go.Figure()
    for tipo in ["Pública", "Privada"]:
        d = sub[sub["ESCOLA_TIPO"] == tipo]
        fig.add_trace(go.Bar(
            x=d["REGIAO_NOME"],
            y=d["INSE_MEDIO"],
            name=tipo,
            marker_color=CORES_TIPO_DARK[tipo],
            opacity=0.85,
            text=d["INSE_MEDIO"].round(2),
            textposition="outside",
            textfont=dict(color=DB_TEXT, size=11),
            hovertemplate=(
                f"<b>%{{x}} · {tipo} · {ano}</b><br>"
                "INSE Médio: %{y:.2f}<extra></extra>"
            ),
        ))

    fig.update_layout(
        paper_bgcolor=DB_BG,
        plot_bgcolor=DB_CARD,
        title=dict(
            text=(
                f"<b style='font-size:18px'>INSE Médio por Região — {ano}</b><br>"
                f"<span style='color:{DB_SUBTEXT};font-size:13px'>"
                "Filtro: tipo de escola (Pública vs. Privada)</span>"
            ),
            x=0.03, y=0.95,
            font=dict(color=DB_TEXT, family="Inter, Arial, sans-serif"),
        ),
        barmode="group",
        xaxis=dict(
            title=dict(text="Região", font=dict(color=DB_SUBTEXT, size=12)),
            tickfont=dict(color=DB_SUBTEXT, size=12),
            gridcolor=DB_GRID, linecolor=DB_GRID,
            categoryorder="array",
            categoryarray=REGIOES_ORDEM,
        ),
        yaxis=dict(
            title=dict(text="INSE Médio", font=dict(color=DB_SUBTEXT, size=12)),
            tickfont=dict(color=DB_SUBTEXT, size=11),
            gridcolor=DB_GRID, linecolor=DB_GRID,
            zeroline=False,
            range=[0, inse_reg["INSE_MEDIO"].max() * 1.18],
        ),
        legend=dict(
            title=dict(text="Escola", font=dict(color=DB_TEXT)),
            orientation="h",
            x=0, y=-0.15,
            font=dict(color=DB_TEXT, size=12),
            bgcolor="rgba(0,0,0,0)",
            bordercolor=DB_ORANGE, borderwidth=1,
        ),
        font=DB_FONT,
        height=480,
        margin=dict(l=60, r=40, t=100, b=80),
    )
    return fig

fig3a = barras_inse(2021)
fig3a.show()

fig3b = barras_inse(2023)
fig3b.show()

##Gráfico 4 - Comparativo 2021 × 2023: acertos médios por região e por tipo de escola

In [25]:
acertos_comp = (
    df.groupby(["ANO", "REGIAO_NOME", "ESCOLA_TIPO"])
      .agg(ACERTOS_TOTAIS=("ACERTOS_TOTAIS", "mean"))
      .reset_index()
)

fig4a = go.Figure()

for tipo, dash in [("Pública", "solid"), ("Privada", "dot")]:
    for ano, cor in [(2021, COR_2021), (2023, COR_2023)]:
        d = (
            acertos_comp[
                (acertos_comp["ESCOLA_TIPO"] == tipo) &
                (acertos_comp["ANO"] == ano)
            ].sort_values("REGIAO_NOME")
        )
        fig4a.add_trace(go.Bar(
            x=d["REGIAO_NOME"],
            y=d["ACERTOS_TOTAIS"],
            name=f"{ano} · {tipo}",
            marker_color=cor,
            marker_pattern_shape="" if tipo == "Pública" else "/",
            opacity=0.85,
            legendgroup=f"{ano}-{tipo}",
            text=d["ACERTOS_TOTAIS"].round(1),
            textposition="outside",
            textfont=dict(color=DB_TEXT, size=10),
            hovertemplate=(
                f"<b>%{{x}} · {tipo} · {ano}</b><br>"
                "Acertos médios: %{y:.1f}<extra></extra>"
            ),
        ))

fig4a.update_layout(
    paper_bgcolor=DB_BG,
    plot_bgcolor=DB_CARD,
    title=dict(
        text=(
            "<b style='font-size:18px'>Comparativo 2021 × 2023 — Acertos Médios</b><br>"
            f"<span style='color:{DB_SUBTEXT};font-size:13px'>"
            "Por região e tipo de escola · barras hachuradas = Privada</span>"
        ),
        x=0.03, y=0.95,
        font=dict(color=DB_TEXT, family="Inter, Arial, sans-serif"),
    ),
    barmode="group",
    xaxis=dict(
        title=dict(text="Região", font=dict(color=DB_SUBTEXT, size=12)),
        tickfont=dict(color=DB_SUBTEXT, size=12),
        gridcolor=DB_GRID, linecolor=DB_GRID,
        categoryorder="array",
        categoryarray=REGIOES_ORDEM,
    ),
    yaxis=dict(
        title=dict(text="Acertos Médios (LP + MT)", font=dict(color=DB_SUBTEXT, size=12)),
        tickfont=dict(color=DB_SUBTEXT, size=11),
        gridcolor=DB_GRID, linecolor=DB_GRID,
        zeroline=False,
        range=[0, acertos_comp["ACERTOS_TOTAIS"].max() * 1.18],
    ),
    legend=dict(
        title=dict(text="Ano · Escola", font=dict(color=DB_TEXT)),
        orientation="h",
        x=0, y=-0.15,
        font=dict(color=DB_TEXT, size=12),
        bgcolor="rgba(0,0,0,0)",
        bordercolor=DB_ORANGE, borderwidth=1,
    ),
    font=DB_FONT,
    height=520,
    margin=dict(l=60, r=40, t=100, b=100),
)
fig4a.show()

